# 06 — RAG Evaluation & Observability with Ragas and LangSmith

This notebook implements an **end-to-end evaluation and observability pipeline** for the **Intelligent Educational Assessment System**.

---

### Core Objectives
1. **Reuse Existing RAG Architecture**: Interrogate the production ChromaDB curriculum store (`240 chunks across 14 lessons`) using local Hugging Face sentence embeddings and generate grounded curriculum answers via the existing configured LLM provider (Groq / Gemini / Ollama).
2. **LangSmith Full-Stack Tracing**: Capture trace trees (`curriculum_retrieval` $\rightarrow$ `curriculum_generation` $\rightarrow$ `curriculum_rag_pipeline`), logging execution latencies, retrieved chunk IDs, and trace URLs directly in the notebook without exposing secret credentials.
3. **Ragas 4-Metric Evaluation**:
   - **Context Precision**: Assesses whether retrieved curriculum chunks place the ground-truth information at the highest ranks.
   - **Context Recall**: Evaluates whether all necessary facts from the reference answer are present in the retrieved curriculum context.
   - **Faithfulness**: Measures whether claims in the generated response are strictly grounded in the retrieved text (hallucination detection).
   - **Answer Relevancy**: Quantifies how directly and concisely the response addresses the student's question.
4. **Interactive Visualizations & Analysis**: Aggregate metrics summaries, per-question score profiles, retrieval snippet inspection, and automated artifact export.


## Section A — Environment Setup

In this section, we:
- Locate the project root and load environment variables from `.env`.
- Verify required configurations without leaking secret API keys.
- Check LangSmith tracing toggles and installed package versions.

In [2]:
import sys
from pathlib import Path

# Resolve project root containing scripts/
candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
ROOT_DIR = next((p for p in candidates if (p / "scripts" / "config.py").is_file()), None)
if ROOT_DIR is None:
    raise FileNotFoundError("Could not find the project root containing scripts/config.py")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Import evaluation module utilities
from scripts.evaluation import (
    validate_eval_environment,
    load_and_validate_dataset,
    run_rag_batch,
    evaluate_rag_results,
    aggregate_evaluation_results,
    save_evaluation_outputs,
    setup_ragas_evaluators,
    _LANGSMITH_AVAILABLE,
    LangSmithClient,
)

# Run environment and credentials validation
env_status = validate_eval_environment()

print("=" * 60)
print(f"Project Root: {ROOT_DIR}")
print(f".env File Loaded: {env_status['env_file_exists']}")
print(f"Active LLM Provider: {env_status['llm_provider']}")
print(f"Primary Provider Configured: {env_status['primary_provider_configured']}")
print(f"Groq API Key: {env_status['groq_api_key_masked']}")
print(f"Gemini API Key: {env_status['gemini_api_key_masked']}")
print(f"LangSmith Tracing Enabled: {env_status['langsmith_tracing_enabled']}")
print(f"LangSmith API Key: {env_status['langsmith_api_key_masked']}")
print(f"LangSmith Project: {env_status['langsmith_project']}")
print(f"Vector Database Exists: {env_status['vector_db_exists']} ({env_status['vector_db_path']})")
print("=" * 60)
print("Installed Dependency Versions:")
for pkg, ver in env_status["dependency_versions"].items():
    print(f"  - {pkg}: {ver}")
print("=" * 60)

if not env_status["vector_db_exists"]:
    raise FileNotFoundError("Vector database not found. Please run verify_store.py or ingestion first.")
if not env_status["primary_provider_configured"]:
    raise ValueError("Primary LLM provider is not configured. Please check your .env credentials.")

Project Root: D:\training\SIC\grad\Intelligent-Educational-Assessment-System
.env File Loaded: True
Active LLM Provider: groq
Primary Provider Configured: True
Groq API Key: ****************************************************zaxg
Gemini API Key: ***********************************9aW4
LangSmith Tracing Enabled: True
LangSmith API Key: ***********************************************5fe3
LangSmith Project: educational-assessment
Vector Database Exists: True (D:\training\SIC\grad\Intelligent-Educational-Assessment-System\data\processed\chroma_db)
Installed Dependency Versions:
  - python: 3.10.0
  - ragas: 0.4.3
  - langsmith: 0.14.12
  - langchain-core: 1.7.0
  - chromadb: 1.5.9
  - pandas: 2.3.3


## Section B — Load and Validate the Evaluation Dataset

We load the ground-truth evaluation dataset from `data/evaluation/rag_test_set.json`.

Each item is validated and normalized into the standard evaluation schema:
- `user_input`: The educational query / test question.
- `reference`: The ground truth curriculum reference answer.
- `source_chapter`, `source_page`, and question metadata.

Any missing fields, empty strings, or duplicate questions are explicitly reported rather than silently ignored.

In [4]:
import pandas as pd

# Load dataset using validation utility
valid_cases, invalid_cases = load_and_validate_dataset()

print("=" * 60)
print(f"Total Valid Test Cases:   {len(valid_cases)}")
print(f"Total Excluded/Invalid:   {len(invalid_cases)}")
print("=" * 60)

if invalid_cases:
    print("Excluded Cases Report:")
    for inv in invalid_cases:
        print(f"  * ID: {inv.get('id', 'N/A')} | Reason: {inv.get('exclusion_reason')}")
else:
    print("All 30 test cases in rag_test_set.json passed schema validation!")

# Display preview of first 5 valid test cases
df_preview = pd.DataFrame(valid_cases)[["id", "question", "reference_answer", "source_chapter", "source_page"]]
display(df_preview.head(5))

Total Valid Test Cases:   30
Total Excluded/Invalid:   0
All 30 test cases in rag_test_set.json passed schema validation!
    id  ... source_page
0  q01  ...           5
1  q02  ...          13
2  q03  ...           7
3  q04  ...          27
4  q05  ...          41

[5 rows x 5 columns]


## Section C — Run the Existing RAG Pipeline

For each selected question, we execute the actual system RAG pipeline:
1. **Retrieve Curriculum Context**: Queries the existing ChromaDB vector store (`scripts/generation.py:retrieve_curriculum_context`) for the top-$k$ relevant chunks ($k=4$).
2. **Measure Latencies**: Records retrieval latency, generation latency, and total end-to-end execution latency separately.
3. **Generate Answer**: Formats the retrieved curriculum passages into a prompt and invokes `_send_completion` using the active provider (`Groq` / `openai/gpt-oss-20b`).
4. **Smoke Test Safety**: We first run a configurable **smoke test of 3 questions**. Once inspected, full evaluation across all 30 questions can be triggered by setting `RUN_FULL_EVALUATION = True`.

In [6]:
# ==============================================================================
# Configuration: Smoke Test vs Full Evaluation
# ==============================================================================
# Set RUN_FULL_EVALUATION to True to evaluate the complete 30-question dataset.
# Keep False for the quick 3-question smoke test.
RUN_FULL_EVALUATION = False

selected_cases = valid_cases if RUN_FULL_EVALUATION else valid_cases[:3]
print(f"Selected {len(selected_cases)} test case(s) for execution (Mode: {'FULL EVALUATION' if RUN_FULL_EVALUATION else 'SMOKE TEST'}).")

# Initialize LangSmith client if tracing is enabled
ls_client = LangSmithClient() if _LANGSMITH_AVAILABLE and env_status["langsmith_tracing_enabled"] else None

def progress_logger(idx, total, q):
    print(f"[{idx}/{total}] Processing: {q[:65]}...")

# Run the RAG batch
rag_results = run_rag_batch(
    selected_cases,
    k=4,
    langsmith_client=ls_client,
    progress_callback=progress_logger,
)

print("\nRAG execution completed successfully!")

Selected 3 test case(s) for execution (Mode: SMOKE TEST).
[1/3] Processing: What is Moore's Law?...
[2/3] Processing: What is Artificial Intelligence (AI)?...
[3/3] Processing: What is edge computing?...

RAG execution completed successfully!



Loading weights: 100%|##########| 103/103 [00:00<00:00, 1714.09it/s]

Loading weights: 100%|##########| 103/103 [00:00<00:00, 1687.51it/s]

Loading weights: 100%|##########| 103/103 [00:00<00:00, 2059.13it/s]


## Section D — LangSmith Tracing & Observability

Observability is instrumented directly into the execution path using `@traceable`:
- **`curriculum_retrieval`** (`run_type="retriever"`): Records the query, retrieved chunk IDs, and latency.
- **`curriculum_generation`** (`run_type="llm"`): Records prompt input, response payload, model identifier, and latency.
- **`curriculum_rag_pipeline`** (`run_type="chain"`): The parent run uniting retrieval and generation into a unified trace.

Traces are automatically associated with the project `educational-assessment`. If `LANGSMITH_TRACING=false` is set in `.env`, tracing is gracefully disabled without altering RAG behavior.

In [8]:
# Inspect LangSmith Trace Runs
trace_summary = []
for res in rag_results:
    trace_summary.append({
        "Test ID": res["id"],
        "Question": res["question"][:45] + "...",
        "Run ID": res["run_id"] or "Local (No Trace)",
        "Retrieval (s)": res["retrieval_latency"],
        "Generation (s)": res["generation_latency"],
        "Total (s)": res["total_latency"],
        "Status": res["status"],
        "Trace URL": res["trace_url"] or "N/A",
    })

trace_df = pd.DataFrame(trace_summary)
display(trace_df)

print("\n" + "=" * 80)
print(f"LangSmith Project: {env_status['langsmith_project']}")
print("To inspect traces in your browser:")
print(f"1. Open https://smith.langchain.com")
print(f"2. Navigate to project: '{env_status['langsmith_project']}'")
print(f"3. Click on any Run ID to inspect inputs, retrieved chunks, outputs, and latency spans.")
print("=" * 80)

  Test ID  ...                                          Trace URL
0     q01  ...  https://smith.langchain.com/o/2058d546-e850-45...
1     q02  ...  https://smith.langchain.com (Run ID: 01a17eda-...
2     q03  ...  https://smith.langchain.com (Run ID: 01a17eda-...

[3 rows x 8 columns]

LangSmith Project: educational-assessment
To inspect traces in your browser:
1. Open https://smith.langchain.com
2. Navigate to project: 'educational-assessment'
3. Click on any Run ID to inspect inputs, retrieved chunks, outputs, and latency spans.


## Section E — Ragas Evaluation (4 Core Metrics)

We evaluate the generated RAG outputs using **Ragas 0.4.3**:

| Metric | Type | Purpose |
| :--- | :--- | :--- |
| **Context Precision** | Reference-based | Evaluates whether retrieved curriculum chunks place relevant information at the top ranks. |
| **Context Recall** | Reference-based | Evaluates whether all necessary ground truth claims are present in the retrieved chunks. |
| **Faithfulness** | Context-grounded | Verifies that all claims made in the answer are strictly supported by the retrieved context (no hallucinations). |
| **Answer Relevancy** | Query-grounded | Measures how directly the generated answer addresses the question without superfluous fluff. |

**Evaluator Configuration**:
- **Evaluator LLM**: Groq `openai/gpt-oss-20b` via `ChatOpenAI` wrapped with `bypass_n=True` to conform to Groq API rate limits.
- **Evaluator Embeddings**: Local Hugging Face `sentence-transformers/all-MiniLM-L6-v2` embeddings.

In [10]:
print("Configuring evaluator LLM and Embeddings...")
evaluator_llm, evaluator_emb = setup_ragas_evaluators()

print("Running Ragas evaluation across 4 metrics (this may take 30-90 seconds)...")
results_df = evaluate_rag_results(
    rag_results,
    evaluator_llm=evaluator_llm,
    evaluator_emb=evaluator_emb,
)

print("\nRagas Evaluation Completed!")

Configuring evaluator LLM and Embeddings...
Running Ragas evaluation across 4 metrics (this may take 30-90 seconds)...

Ragas Evaluation Completed!



Loading weights: 100%|##########| 103/103 [00:00<00:00, 1208.87it/s]

Evaluating: 100%|##########| 12/12 [03:29<00:00, 17.44s/it]


## Section F — Display Results Inside the Notebook

Here we analyze the evaluation results:
1. **Per-Question Detailed Table**: Displays questions, answers, chunk counts, latencies, and metric scores.
2. **Aggregate Metrics Summary**: Mean scores with valid denominators.
3. **Visualizations**: Matplotlib plots for aggregate metrics and per-question score profiles.
4. **Retrieval & Failure Analysis**: Weak cases, low scores, and chunk text previews.

In [12]:
import numpy as np

# 1. Per-Question Results Table
display_cols = [
    "id",
    "question",
    "reference",
    "response",
    "num_chunks",
    "retrieval_latency",
    "generation_latency",
    "context_precision",
    "context_recall",
    "faithfulness",
    "answer_relevancy",
    "status",
]
display_df = results_df[[c for c in display_cols if c in results_df.columns]].copy()
display(display_df)

# 2. Aggregate Metrics Summary
summary = aggregate_evaluation_results(results_df)

print("\n" + "=" * 60)
print("AGGREGATE EVALUATION METRICS SUMMARY")
print("=" * 60)
print(f"Total Cases Evaluated: {summary['total_evaluated_cases']}")
for metric, stats in summary["metrics"].items():
    mean_str = f"{stats['mean']:.4f}" if stats["mean"] is not None else "N/A"
    print(f"  * {metric.replace('_', ' ').title():<22}: {mean_str} (Valid: {stats['valid_count']}, Failed/Missing: {stats['missing_or_failed_count']})")

print("\nLatency Summary:")
for lat_name, lat_val in summary["latency_summary"].items():
    print(f"  * {lat_name.replace('_', ' ').title()}: {lat_val:.3f}s")
print("=" * 60)

    id                               question  ... answer_relevancy     status
0  q01                   What is Moore's Law?  ...         0.903959  evaluated
1  q02  What is Artificial Intelligence (AI)?  ...         0.877568  evaluated
2  q03                What is edge computing?  ...         1.000000  evaluated

[3 rows x 12 columns]

AGGREGATE EVALUATION METRICS SUMMARY
Total Cases Evaluated: 3
  * Context Precision     : 0.9444 (Valid: 3, Failed/Missing: 0)
  * Context Recall        : 1.0000 (Valid: 3, Failed/Missing: 0)
  * Faithfulness          : 1.0000 (Valid: 3, Failed/Missing: 0)
  * Answer Relevancy      : 0.9272 (Valid: 3, Failed/Missing: 0)

Latency Summary:
  * Mean Retrieval Latency: 5.606s
  * Mean Generation Latency: 1.169s
  * Mean Total Latency: 7.944s


In [13]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 3. Bar Chart: Aggregate Ragas Scores
metrics_dict = summary["metrics"]
metric_names = [m.replace("_", " ").title() for m in metrics_dict.keys()]
metric_scores = [metrics_dict[m]["mean"] if metrics_dict[m]["mean"] is not None else 0.0 for m in metrics_dict.keys()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Aggregate Metrics
colors = ["#2b5c8f", "#3a86c8", "#48bfe3", "#56cfe1"]
bars = ax1.bar(metric_names, metric_scores, color=colors, width=0.55, edgecolor="#1d3557", linewidth=1.2)
ax1.set_ylim(0.0, 1.1)
ax1.set_title("Aggregate Ragas Metrics (Mean Scores)", fontsize=13, fontweight="bold", pad=12)
ax1.set_ylabel("Score (0.0 to 1.0)", fontsize=11)
ax1.grid(axis="y", linestyle="--", alpha=0.5)

for bar, score in zip(bars, metric_scores):
    height = bar.get_height()
    ax1.annotate(f"{score:.3f}",
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 4),
                 textcoords="offset points",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")

# Plot 2: Per-Question Latency Breakdown
q_ids = results_df["id"].tolist()
ret_lat = results_df["retrieval_latency"].tolist()
gen_lat = results_df["generation_latency"].tolist()
x = np.arange(len(q_ids))
width = 0.35

ax2.bar(x - width/2, ret_lat, width, label="Retrieval Latency", color="#64dfdf", edgecolor="#333")
ax2.bar(x + width/2, gen_lat, width, label="Generation Latency", color="#ffb703", edgecolor="#333")
ax2.set_xlabel("Question ID", fontsize=11)
ax2.set_ylabel("Latency (seconds)", fontsize=11)
ax2.set_title("Per-Question Execution Latencies", fontsize=13, fontweight="bold", pad=12)
ax2.set_xticks(x)
ax2.set_xticklabels(q_ids)
ax2.legend()
ax2.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plot_path = ROOT_DIR / "outputs" / "evaluation" / "ragas_metrics_plot.png"
plot_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Metrics plot successfully saved to: {plot_path.relative_to(ROOT_DIR)}")
plt.close(fig)

Metrics plot successfully saved to: outputs\evaluation\ragas_metrics_plot.png


In [14]:
# 4. Retrieval Inspection Table & Failure Analysis
print("RETRIEVAL INSPECTION (Safe Previews Without Full Document Dumps)")
print("=" * 80)
for res in rag_results[:3]:
    print(f"\nQuestion ID [{res['id']}]: {res['question']}")
    print(f"Referenced Chunks: {res['chunk_ids']}")
    for i, chunk_text in enumerate(res["retrieved_contexts"][:2]):
        snippet = chunk_text.strip().replace('\n', ' ')[:100]
        print(f"  [Passage {i+1} Preview]: {snippet}...")
print("=" * 80)

# Failure & Weakness Analysis
print("\nFAILURE & WEAKNESS ANALYSIS:")
weak_cases = results_df[
    (results_df["faithfulness"] < 0.7) |
    (results_df["answer_relevancy"] < 0.7) |
    (results_df["context_precision"] < 0.7) |
    (results_df["status"] != "evaluated")
]

if not weak_cases.empty:
    print(f"Detected {len(weak_cases)} case(s) requiring attention:")
    display(weak_cases[["id", "question", "faithfulness", "answer_relevancy", "context_precision", "status", "error"]])
else:
    print("All evaluated cases achieved strong scores (>= 0.70) across evaluated metrics!")

RETRIEVAL INSPECTION (Safe Previews Without Full Document Dumps)

Question ID [q01]: What is Moore's Law?
Referenced Chunks: ['1-1-010', '1-1-012', '1-1-005', '1-1-004']
  [Passage 1 Preview]: B Birth of the computer → Commercialization of the Internet → Rise of smartphones → Spread of cloud ...
  [Passage 2 Preview]: (2)  A Correct definition of Moore's Law. Therefore, ○.  B It has been said to be approaching a phys...

Question ID [q02]: What is Artificial Intelligence (AI)?
Referenced Chunks: ['1-2-002', '1-2-012', '1-2-003', '1-2-001']
  [Passage 1 Preview]: learning, and generative AI related?  ✎  Explore · In Pairs  With a partner, list three tasks a phon...
  [Passage 2 Preview]: ❓  Lesson Question — Answered  The lesson question asked what AI is and how machine learning, deep  ...

Question ID [q03]: What is edge computing?
Referenced Chunks: ['1-1-007', '1-1-009', '1-1-015', '1-1-008']
  [Passage 1 Preview]: Five social changes driven by information technology.  [Side note] KE

## Section G — Save Outputs

We persist all evaluation artifacts to `outputs/evaluation/`:
- **`ragas_results.csv`**: Tabular per-question results with latencies and scores.
- **`ragas_summary.json`**: Aggregate scores, valid denominators, and execution metadata.
- **`ragas_results.json`**: Full detailed evaluation records.
- **`run_manifest.jsonl`**: JSON Lines run log with trace URLs, sanitized of secrets.

In [16]:
# Save outputs to outputs/evaluation/
out_files = save_evaluation_outputs(results_df, summary, rag_results)

print("=" * 60)
print("EVALUATION ARTIFACTS PERSISTED SUCCESSFULLY")
print("=" * 60)
for key, filepath in out_files.items():
    print(f"  * {key.upper():<15}: {filepath.relative_to(ROOT_DIR)} (Size: {filepath.stat().st_size} bytes)")
print("=" * 60)
print("\nNotebook execution complete!")

EVALUATION ARTIFACTS PERSISTED SUCCESSFULLY
  * CSV            : outputs\evaluation\ragas_results.csv (Size: 1778 bytes)
  * SUMMARY_JSON   : outputs\evaluation\ragas_summary.json (Size: 923 bytes)
  * RESULTS_JSON   : outputs\evaluation\ragas_results.json (Size: 2657 bytes)
  * MANIFEST_JSONL : outputs\evaluation\run_manifest.jsonl (Size: 12064 bytes)

Notebook execution complete!
